# Aula 9 — Prática: da lista de genes às vias metabólicas

**Introdução à Bioinformática — PPGED/IME** · Momento 2 (≈ 70 min)

**Pergunta da prática:**
> Células expostas a **pouco oxigênio (hipóxia)** ligam um conjunto de genes. *Que processos biológicos esses genes contam juntos?* E os **metabólitos** medidos nas mesmas condições confirmam a história?

Por que hipóxia? Porque é um problema operacional real: **voo em alta altitude, tropas de montanha, submarinos, mergulho, uso prolongado de máscaras e roupas de proteção QBRN**. A resposta celular à falta de oxigênio é controlada por um "interruptor" molecular, o fator **HIF‑1** (*Hypoxia‑Inducible Factor 1*), que reprograma o metabolismo da célula.

### O que você vai fazer

| Parte | O que acontece | Ferramenta | Tempo |
|---|---|---|---|
| P1 | Preparar o ambiente e testar a conexão | Python, `requests` | 5 min |
| P2 | Conhecer os dados (genes e metabólitos) | `pandas` | 5 min |
| P3 | Resolver o problema dos identificadores | KEGG API | 5 min |
| P4 | Fazer uma análise de enriquecimento **"na mão"** | `scipy` + KEGG | 15 min |
| P5 | Controle negativo: uma lista **aleatória** | idem | 5 min |
| P6 | Repetir com uma ferramenta profissional | **g:Profiler** | 8 min |
| P7 | Consultar um banco curado de vias | **Reactome** | 7 min |
| P8 | Integrar genes **e** metabólitos num mapa de via | **KEGG** | 10 min |
| P9 | Ver as proteínas como **rede** | **STRING** + `networkx` | 8 min |
| P10 | Salvar os resultados | `pandas` | 2 min |

### Como usar este caderno
1. Abra no **Google Colab** (Arquivo → Abrir notebook → Upload).
2. Clique em cada célula de código e aperte **▶** (ou `Shift + Enter`), **na ordem**, de cima para baixo.
3. **Leia** o texto antes de cada célula e as perguntas depois dela. O objetivo não é programar: é **executar, observar e interpretar**.
4. Linhas que começam com `#` são **comentários**: explicações para humanos, que o computador ignora.

> **Rede bloqueada?** Este caderno conversa com quatro serviços públicos na internet (KEGG, g:Profiler, Reactome e STRING). Algumas redes institucionais bloqueiam esses endereços. A célula P1 testa todos eles; se algum falhar, rode o caderno de casa ou pelo celular roteado. **Teste na véspera.**

---
# P1. Preparar o ambiente (≈ 5 min)

O Colab já vem com quase tudo instalado. Vamos usar estas **bibliotecas** (pacotes de código prontos, como "caixas de ferramentas"):

| Biblioteca | Para que serve aqui |
|---|---|
| `requests` | Fazer pedidos a sites e **APIs** (lembre da Aula 1: a API é o "garçom" que leva seu pedido ao banco de dados e traz a resposta) |
| `pandas` | Trabalhar com **tabelas** (como uma planilha de Excel dentro do Python) |
| `numpy` | Contas numéricas |
| `scipy` | Estatística — aqui, o **teste hipergeométrico** |
| `matplotlib` | Gráficos |
| `networkx` | Construir e medir **redes** (grafos) |
| `gprofiler-official` | Cliente oficial em Python do serviço **g:Profiler** — é a única coisa que precisamos instalar |

In [ ]:
# O "!" diz ao Colab: rode isto no terminal, não no Python.
# "-q" = quiet (silencioso), só para não poluir a tela.
!pip install -q gprofiler-official

import io        # trata um texto como se fosse um arquivo (útil para ler tabelas baixadas)
import time      # permite esperar alguns segundos entre tentativas
import random    # sorteios (vamos usar no controle negativo)

import requests                  # conversa com APIs na internet
import numpy as np               # contas numéricas
import pandas as pd              # tabelas
import matplotlib.pyplot as plt  # gráficos
import networkx as nx            # redes
from scipy.stats import hypergeom                  # distribuição hipergeométrica
from IPython.display import display, Image, Markdown  # mostrar tabelas, imagens e texto formatado

pd.set_option("display.max_colwidth", 90)   # mostra textos mais longos nas tabelas

# Todo pedido a uma API leva uma "identificação" educada de quem está pedindo.
CABECALHO = {"User-Agent": "Aula9-Bioinformatica-PPGED-IME (uso didatico)"}

def baixar(url, metodo="GET", tentativas=3, cabecalhos_extra=None, **kwargs):
    """Faz um pedido a uma API e devolve a resposta.

    Se a internet falhar, tenta de novo algumas vezes (com espera crescente),
    em vez de travar o caderno com uma mensagem de erro assustadora.
    """
    cab = dict(CABECALHO)
    if cabecalhos_extra:
        cab.update(cabecalhos_extra)
    for tentativa in range(1, tentativas + 1):
        try:
            r = requests.request(metodo, url, headers=cab, timeout=90, **kwargs)
            if r.status_code == 200:          # 200 = "deu certo" no protocolo da web
                return r
            print(f"  aviso: o servidor respondeu com código {r.status_code} (tentativa {tentativa})")
        except requests.RequestException as erro:
            print(f"  aviso: falha de conexão ({type(erro).__name__}) (tentativa {tentativa})")
        time.sleep(2 * tentativa)
    raise RuntimeError(f"Não consegui acessar {url[:70]}... Verifique sua conexão ou se a rede bloqueia o site.")

print("Ambiente pronto!")

In [ ]:
# Teste de conexão com os quatro serviços que vamos usar.
servicos = {
    "KEGG (vias e metabólitos)":      "https://rest.kegg.jp/info/hsa",
    "g:Profiler (enriquecimento)":    "https://biit.cs.ut.ee/gprofiler/",
    "Reactome (vias curadas)":        "https://reactome.org/ContentService/data/database/version",
    "STRING (redes de proteínas)":    "https://string-db.org/api/json/version",
}
for nome, url in servicos.items():
    try:
        baixar(url, tentativas=1)
        print(f"OK     {nome}")
    except RuntimeError:
        print(f"FALHOU {nome}  → esta parte da prática não vai funcionar nesta rede")

---
# P2. Os dados (≈ 5 min)

## P2.1 A lista de genes

Abaixo está uma lista de **36 genes** que, segundo a literatura, são **ligados pelo HIF‑1 quando falta oxigênio** (os "genes‑alvo clássicos do HIF"). Pense nela como o resultado típico de um experimento de transcriptômica (Aula 4) ou proteômica (Aula 5): *"estes genes/proteínas aumentaram na hipóxia"*.

Os genes estão escritos pelo **símbolo oficial** (nome curto aprovado pelo comitê internacional HGNC). Agrupamos com comentários só para você se orientar — **o computador não sabe desses grupos**; ele vai ter que descobri‑los sozinho. É exatamente isso que queremos testar.

> **Honestidade sobre os dados:** esta é uma lista *curada da literatura*, não saída de um único experimento. Isso é ótimo para aprender (sabemos a resposta esperada), mas lembre: num experimento real, a lista vem com ruído, falsos positivos e genes que ninguém sabe o que fazem.

In [ ]:
GENES_HIPOXIA = [
    # Entrada de glicose e glicólise (quebra do açúcar sem oxigênio)
    "SLC2A1", "SLC2A3", "HK2", "GPI", "PFKL", "PFKP", "PFKFB3", "ALDOA",
    "ALDOC", "TPI1", "PGK1", "PGAM1", "ENO1", "PKM", "LDHA", "PDK1",
    # Formação de vasos e oferta de oxigênio ao tecido
    "VEGFA", "ADM", "ANGPTL4", "EPO",
    # Controle do pH (a glicólise acidifica a célula)
    "CA9",
    # Sensores de oxigênio (as enzimas que "medem" O2 e regulam o próprio HIF)
    "EGLN1", "EGLN3",
    # Sobrevivência, autofagia de mitocôndrias e controle do crescimento
    "BNIP3", "BNIP3L", "DDIT4", "NDRG1",
    # Matriz extracelular e colágeno (hidroxilases que também usam O2)
    "P4HA1", "P4HA2", "PLOD2", "LOX",
    # Regulação da transcrição e da cromatina (lembre da Aula 6!)
    "BHLHE40", "KDM3A", "ANKRD37", "MXI1",
    # Dobramento de proteínas no retículo endoplasmático
    "ERO1A",
]
print(f"{len(GENES_HIPOXIA)} genes na lista.")

## P2.2 Os metabólitos (camada metabolômica)

Agora imagine que, no mesmo experimento, um colega mediu **metabólitos** por espectrometria de massas (a mesma tecnologia da Aula 5, aplicada a moléculas pequenas). O resultado é uma tabela com o **log₂ da mudança** (*log₂ fold change*, lembre da Aula 4): valor positivo = aumentou na hipóxia; negativo = diminuiu.

Cada metabólito já vem com seu **identificador KEGG Compound** (número começando com **C**). Conseguir esse identificador é, na vida real, uma das etapas mais difíceis da metabolômica — falaremos disso no Momento 3.

> **Atenção:** os valores abaixo são **ilustrativos**, escolhidos para serem coerentes com a fisiologia conhecida da hipóxia (mais lactato, menos glicose, acúmulo de succinato). Dados reais de metabolômica estão em repositórios públicos como **MetaboLights** (EBI) e **Metabolomics Workbench** (NIH), citados no roteiro.

In [ ]:
METABOLITOS = pd.DataFrame({
    "metabolito": ["D-Glicose", "Piruvato", "L-Lactato", "L-Alanina",
                   "Citrato", "Succinato", "2-Oxoglutarato"],
    "kegg_id":    ["C00031",    "C00022",   "C00186",    "C00041",
                   "C00158",    "C00042",    "C00026"],
    "log2FC":     [-1.0,          0.6,        2.0,         0.8,
                   -0.9,          1.4,       -0.7],
})
display(METABOLITOS)

**Pergunta P2** — Sem rodar nada ainda: olhando os comentários da lista de genes e a tabela de metabólitos, qual processo metabólico você *aposta* que vai aparecer como o principal? Anote sua aposta.

---
# P3. O problema dos identificadores (≈ 5 min)

Cada banco de dados dá **seu próprio "RG"** a um mesmo gene. O gene da lactato desidrogenase A, por exemplo, é:

| Banco | Identificador |
|---|---|
| HGNC (símbolo) | `LDHA` |
| NCBI Gene (Entrez) | `3939` |
| Ensembl | `ENSG00000134333` |
| UniProt (proteína) | `P00338` |
| KEGG | `hsa:3939` (*hsa* = *Homo sapiens*) |

Antes de qualquer análise de vias, é preciso **traduzir** sua lista para a "língua" do banco que você vai consultar. Essa etapa se chama **mapeamento de identificadores** (*ID mapping*) e é onde muitos genes "somem" silenciosamente — por isso sempre conferimos quantos foram encontrados.

Vamos baixar do **KEGG** a lista completa de genes humanos (≈ 25 mil linhas) e montar nosso "dicionário de tradução".

**Sobre o KEGG** (*Kyoto Encyclopedia of Genes and Genomes*, Japão, desde 1995): é uma enciclopédia que liga genes, enzimas, reações químicas e metabólitos em **mapas de vias desenhados à mão** por curadores. Sua **API REST** (endereços que começam com `rest.kegg.jp`) é gratuita para uso acadêmico. Os comandos que usaremos:

| Comando da API | O que devolve |
|---|---|
| `/list/hsa` | Todos os genes humanos e seus símbolos |
| `/list/pathway/hsa` | Todas as vias humanas e seus nomes |
| `/link/pathway/hsa` | Qual gene participa de qual via |
| `/link/pathway/cpd:C00186` | Em que vias participa um metabólito |
| `/get/hsa00010/image` | A imagem do mapa de uma via |

In [ ]:
# 1) Baixa a lista de genes humanos do KEGG
r = baixar("https://rest.kegg.jp/list/hsa")

# Cada linha vem separada por TAB (\t). A última coluna tem: "SÍMBOLO, SINÔNIMO1, ...; descrição"
simbolo_para_kegg = {}     # dicionário: símbolo oficial -> identificador KEGG
sinonimo_para_kegg = {}    # dicionário: nomes antigos/alternativos -> identificador KEGG
for linha in r.text.strip().split("\n"):
    partes = linha.split("\t")
    kegg_id = partes[0]                                   # ex.: "hsa:3939"
    nomes = partes[-1].split(";")[0]                      # ex.: "LDHA, GSD11, LDHM, PIG19"
    lista_nomes = [n.strip() for n in nomes.split(",") if n.strip()]
    if not lista_nomes:
        continue
    simbolo_para_kegg.setdefault(lista_nomes[0], kegg_id)  # o primeiro nome é o símbolo oficial
    for sinonimo in lista_nomes[1:]:
        sinonimo_para_kegg.setdefault(sinonimo, kegg_id)

print(f"Genes humanos no KEGG: {len(simbolo_para_kegg):,}")

# 2) Traduz a nossa lista (primeiro pelo símbolo oficial; se falhar, tenta pelos sinônimos)
def traduzir(simbolo):
    return simbolo_para_kegg.get(simbolo) or sinonimo_para_kegg.get(simbolo)

traducao = pd.DataFrame({"simbolo": GENES_HIPOXIA})
traducao["kegg_id"] = traducao["simbolo"].apply(traduzir)
display(traducao.head(8))

nao_encontrados = traducao[traducao["kegg_id"].isna()]["simbolo"].tolist()
print(f"Traduzidos: {traducao['kegg_id'].notna().sum()} de {len(traducao)}")
print("Não encontrados:", nao_encontrados if nao_encontrados else "nenhum")

**Pergunta P3** — Por que guardamos também os **sinônimos**? (Dica: o gene `ERO1A` se chamava `ERO1L` até 2016. Muitos artigos e planilhas antigas ainda usam o nome velho.)

---
# P4. Análise de enriquecimento "na mão" (≈ 15 min)

## P4.1 A lógica da urna

Pense numa **urna** com todas as bolinhas do jogo:

- **N** = total de genes humanos que têm alguma via no KEGG (o **universo**, ou *background*);
- **K** = quantos desses genes pertencem à via que estamos testando (as bolinhas **vermelhas**);
- **n** = tamanho da nossa lista (quantas bolinhas **tiramos** da urna);
- **k** = quantos genes da nossa lista caem na via (quantas **vermelhas** saíram).

A pergunta estatística é: *se eu tirasse **n** bolinhas **ao acaso**, qual a chance de sair **k ou mais** vermelhas?* Essa chance é o **valor‑p** do **teste hipergeométrico** (equivalente ao teste exato de Fisher unilateral). Se for muito pequena, concluímos que a via está **super‑representada** (*enriquecida*) na lista — o "acaso" explica mal o que vimos.

Primeiro baixamos as duas tabelas do KEGG: **gene → via** e **nome das vias**.

In [ ]:
# Tabela gene -> via (≈ 40 mil linhas). Ex.: "hsa:3939   path:hsa00010"
r = baixar("https://rest.kegg.jp/link/pathway/hsa")
gene_via = pd.read_csv(io.StringIO(r.text), sep="\t", header=None, names=["gene", "via"])
gene_via["via"] = gene_via["via"].str.replace("path:", "", regex=False)

# Nomes das vias. Ex.: "hsa00010   Glycolysis / Gluconeogenesis - Homo sapiens (human)"
r = baixar("https://rest.kegg.jp/list/pathway/hsa")
nomes_vias = pd.read_csv(io.StringIO(r.text), sep="\t", header=None, names=["via", "nome"])
nomes_vias["via"] = nomes_vias["via"].str.replace("path:", "", regex=False)
nomes_vias["nome"] = nomes_vias["nome"].str.replace(" - Homo sapiens (human)", "", regex=False)
NOME_DA_VIA = dict(zip(nomes_vias["via"], nomes_vias["nome"]))

# O universo: todos os genes com pelo menos uma via
UNIVERSO = set(gene_via["gene"])
# Para cada via, o conjunto de genes que a compõem
GENES_DA_VIA = gene_via.groupby("via")["gene"].apply(set).to_dict()

print(f"Vias humanas no KEGG: {len(GENES_DA_VIA)}")
print(f"Universo (genes com alguma via): N = {len(UNIVERSO):,}")

## P4.2 Um teste, passo a passo: a glicólise (`hsa00010`)

In [ ]:
lista_kegg = set(traducao["kegg_id"].dropna()) & UNIVERSO   # só genes que estão no universo

VIA = "hsa00010"
N = len(UNIVERSO)                     # bolinhas na urna
K = len(GENES_DA_VIA[VIA])            # bolinhas vermelhas (genes da via)
n = len(lista_kegg)                   # bolinhas que tiramos (nossa lista)
genes_em_comum = lista_kegg & GENES_DA_VIA[VIA]
k = len(genes_em_comum)               # vermelhas que saíram

esperado = n * K / N                  # quantas vermelhas esperaríamos só por acaso
# P(X >= k) = 1 - P(X <= k-1). A função "sf" (survival function) calcula isso direto.
p = hypergeom.sf(k - 1, N, K, n)

print(f"Via: {VIA} — {NOME_DA_VIA.get(VIA)}")
print(f"N = {N:,} | K = {K} | n = {n} | k = {k}")
print(f"Esperado por acaso: {esperado:.2f} genes   |   Observado: {k} genes")
print(f"Enriquecimento: {k / esperado:.0f} vezes mais do que o acaso")
print(f"Valor-p hipergeométrico: {p:.2e}")
simbolo_de = dict(zip(traducao["kegg_id"], traducao["simbolo"]))
print("Genes em comum:", ", ".join(sorted(simbolo_de[g] for g in genes_em_comum)))

**Pergunta P4.1** — Quantos genes da glicólise esperaríamos encontrar por puro acaso? Quantos encontramos? Em palavras, o que significa um valor‑p da ordem de 10⁻²⁰?

## P4.3 Testando **todas** as vias e corrigindo para testes múltiplos

Agora repetimos o teste para as ~370 vias humanas. Problema: com 370 testes a 5% de significância, esperamos ~18 "falsos alarmes" mesmo com uma lista sem sentido. Por isso aplicamos a **correção de Benjamini‑Hochberg**, que transforma cada valor‑p num **FDR** (*False Discovery Rate*, taxa de falsas descobertas) — a mesma ideia da Aula 4 e da Aula 5.

In [ ]:
def benjamini_hochberg(pvalores):
    """Correção de Benjamini-Hochberg (FDR), escrita por extenso para ser lida."""
    p = np.asarray(pvalores, dtype=float)
    m = len(p)
    ordem = np.argsort(p)                         # do menor para o maior valor-p
    ajustado = p[ordem] * m / np.arange(1, m + 1)  # p * (nº de testes / posição)
    ajustado = np.minimum.accumulate(ajustado[::-1])[::-1]  # garante que não diminua ao subir
    resultado = np.empty(m)
    resultado[ordem] = np.minimum(ajustado, 1.0)
    return resultado

def ora_kegg(lista_ids, tamanho_min=5, tamanho_max=500):
    """Análise de super-representação (ORA) da lista contra todas as vias KEGG."""
    lista = set(lista_ids) & UNIVERSO
    N, n = len(UNIVERSO), len(lista)
    linhas = []
    for via, genes in GENES_DA_VIA.items():
        K = len(genes)
        if not (tamanho_min <= K <= tamanho_max):   # ignora vias minúsculas ou gigantes
            continue
        comum = lista & genes
        k = len(comum)
        p = hypergeom.sf(k - 1, N, K, n) if k > 0 else 1.0
        linhas.append({
            "via": via,
            "nome": NOME_DA_VIA.get(via, via),
            "k (na lista)": k,
            "K (na via)": K,
            "razão (k/esperado)": round(k / (n * K / N), 1) if k else 0.0,
            "p": p,
            "genes": ", ".join(sorted(simbolo_de.get(g, g) for g in comum)),
        })
    tabela = pd.DataFrame(linhas)
    tabela["FDR"] = benjamini_hochberg(tabela["p"])
    return tabela.sort_values("p").reset_index(drop=True)

ora = ora_kegg(lista_kegg)
significativas = ora[ora["FDR"] < 0.05]
print(f"Vias testadas: {len(ora)} | Vias com FDR < 5%: {len(significativas)}")
display(significativas.head(15)[["via", "nome", "k (na lista)", "K (na via)",
                                   "razão (k/esperado)", "FDR", "genes"]])

In [ ]:
# Gráfico de pontos (dot plot): o formato mais comum em artigos para mostrar enriquecimento
top = significativas.head(12).iloc[::-1]      # inverte para a mais significativa ficar no topo
fig, ax = plt.subplots(figsize=(8, 5))
pontos = ax.scatter(-np.log10(top["FDR"]), top["nome"],
                    s=top["k (na lista)"] * 40,            # tamanho = nº de genes da lista
                    c=top["razão (k/esperado)"], cmap="viridis")
ax.axvline(-np.log10(0.05), ls="--", c="grey", lw=1)       # linha do FDR = 5%
ax.set_xlabel("−log10(FDR)   (mais à direita = mais significativo)")
ax.set_title("Vias KEGG enriquecidas na lista de hipóxia")
plt.colorbar(pontos, label="enriquecimento (vezes o acaso)")
plt.tight_layout()
plt.show()

**Pergunta P4.2** — Sua aposta da P2 se confirmou? Procure na tabela a via **"HIF-1 signaling pathway"** (`hsa04066`). Por que é *esperado* — e até um pouco *circular* — que ela apareça? (Dica: de onde vieram os genes da nossa lista e de onde vieram os genes que o curador do KEGG pôs nessa via?)

**Pergunta P4.3** — Há vias de **doenças** ou **câncer** na lista? Isso quer dizer que a hipóxia "causa câncer"? Discuta o que significa um gene participar de várias vias.

---
# P5. Controle negativo: e se a lista fosse aleatória? (≈ 5 min)

Um bom engenheiro testa o instrumento com uma amostra em que **ele sabe que não deveria haver sinal**. Vamos sortear genes ao acaso do universo, do **mesmo tamanho** da nossa lista, e rodar exatamente a mesma análise.

In [ ]:
random.seed(2026)   # "semente": faz o sorteio ser o mesmo para todos na turma
lista_aleatoria = random.sample(sorted(UNIVERSO), len(lista_kegg))

ora_aleatoria = ora_kegg(lista_aleatoria)
print(f"Lista aleatória — vias com FDR < 5%: {(ora_aleatoria['FDR'] < 0.05).sum()}")
print(f"Lista aleatória — vias com p < 5% SEM correção: {(ora_aleatoria['p'] < 0.05).sum()}")
display(ora_aleatoria.head(5)[["nome", "k (na lista)", "K (na via)", "p", "FDR"]])

**Pergunta P5** — Sem correção, quantas vias "deram significativas" na lista aleatória? E com FDR? Explique, com suas palavras, por que **nunca** se deve relatar valores‑p sem correção quando se testam centenas de vias. Troque a semente (`2026`) por outro número e rode de novo.

---
# P6. A ferramenta profissional: g:Profiler (≈ 8 min)

O que fizemos "na mão" é o coração de ferramentas como o **g:Profiler**.

**Sobre o g:Profiler** (Universidade de Tartu, Estônia; <https://biit.cs.ut.ee/gprofiler>): serviço web gratuito e de código aberto que faz análise de enriquecimento contra **várias fontes ao mesmo tempo**: Gene Ontology (GO), KEGG, Reactome, WikiPathways, fatores de transcrição, doenças humanas etc. Ele **traduz identificadores sozinho** (aceita símbolos, Ensembl, UniProt…), atualiza os bancos a cada poucos meses e usa uma correção para testes múltiplos própria, a **g:SCS**, desenhada para a estrutura hierárquica do GO. Tem interface web (para clicar) e clientes para Python e R (para automatizar).

Fontes que vamos pedir:

| Código | Fonte | O que contém |
|---|---|---|
| `GO:BP` | Gene Ontology — Processo Biológico | Vocabulário controlado de "o que o gene ajuda a fazer" (ex.: *resposta à hipóxia*) |
| `KEGG` | KEGG Pathway | Mapas de vias desenhados por curadores |
| `REAC` | Reactome | Vias descritas reação por reação |
| `WP` | WikiPathways | Vias mantidas pela comunidade, estilo Wikipédia |

In [ ]:
from gprofiler import GProfiler

gp = GProfiler(return_dataframe=True, user_agent="Aula9-PPGED-IME")
res_gp = gp.profile(
    organism="hsapiens",                       # Homo sapiens
    query=GENES_HIPOXIA,                       # nossa lista (pode ser símbolos!)
    sources=["GO:BP", "KEGG", "REAC", "WP"],   # fontes de anotação
    user_threshold=0.05,                       # limiar de significância
    significance_threshold_method="g_SCS",     # correção para testes múltiplos
    no_evidences=False,                        # traz quais genes caíram em cada termo
)
res_gp["genes"] = res_gp["intersections"].apply(lambda g: ", ".join(sorted(g)))

print("Termos significativos por fonte:")
print(res_gp["source"].value_counts().to_string())

for fonte in ["GO:BP", "KEGG", "REAC"]:
    print(f"\n=== {fonte}: 5 termos mais significativos ===")
    display(res_gp[res_gp["source"] == fonte]
            .head(5)[["native", "name", "intersection_size", "term_size", "p_value", "genes"]])

In [ ]:
# Comparação: as vias KEGG que nós achamos "na mão" batem com as do g:Profiler?
kegg_gp = set(res_gp.loc[res_gp["source"] == "KEGG", "native"].str.replace("KEGG:", "hsa"))
kegg_nos = set(significativas["via"])
print(f"Vias KEGG significativas — nossa análise: {len(kegg_nos)} | g:Profiler: {len(kegg_gp)}")
print(f"Em comum: {len(kegg_nos & kegg_gp)}")
print("Só na nossa:", sorted(NOME_DA_VIA.get(v, v) for v in kegg_nos - kegg_gp)[:8])
print("Só no g:Profiler:", sorted(NOME_DA_VIA.get(v, v) for v in kegg_gp - kegg_nos)[:8])

**Pergunta P6.1** — Os resultados são **parecidos, mas não idênticos**. Liste três motivos possíveis. (Dicas: versão do banco, definição do universo, método de correção, filtro de tamanho de via.) Isso é o mesmo recado da Aula 5: **descreva suas escolhas no método do artigo**.

**Pergunta P6.2** — Olhe os termos do **GO:BP**. Muitos parecem dizer a mesma coisa com palavras diferentes (*"glycolytic process"*, *"ATP generation from ADP"*, *"pyruvate metabolic process"*…). Por quê? (Dica: o GO é uma **hierarquia**; um termo específico está "dentro" de termos mais gerais.)

---
# P7. Reactome: o banco curado reação por reação (≈ 7 min)

**Sobre o Reactome** (<https://reactome.org>; consórcio OICR‑Canadá, EMBL‑EBI, NYU e outros): banco **aberto e gratuito** de vias humanas em que cada passo é uma **reação** revisada por especialistas e ligada às publicações que a comprovam. As vias são organizadas em **hierarquia** (ex.: *Metabolismo → Metabolismo de carboidratos → Glicólise*). Tem um **Serviço de Análise** que recebe uma lista e devolve o enriquecimento, e um navegador gráfico (*Pathway Browser*) em que você vê sua lista pintada sobre as vias.

Vamos enviar nossa lista à API de análise (`AnalysisService`).

In [ ]:
url = ("https://reactome.org/AnalysisService/identifiers/projection"
       "?pageSize=20&page=1&sortBy=ENTITIES_PVALUE&order=ASC&resource=TOTAL")
r = baixar(url, metodo="POST",
           data="\n".join(GENES_HIPOXIA),                    # um gene por linha
           cabecalhos_extra={"Content-Type": "text/plain"})
resposta = r.json()

token = resposta["summary"]["token"]   # "senha" temporária que guarda sua análise no servidor
reactome = pd.DataFrame([{
    "id": v["stId"],
    "via": v["name"],
    "encontrados": v["entities"]["found"],
    "total na via": v["entities"]["total"],
    "p": v["entities"]["pValue"],
    "FDR": v["entities"]["fdr"],
} for v in resposta["pathways"]])
display(reactome.head(12))

nao_achados = resposta.get("identifiersNotFound", 0)
print(f"Identificadores não reconhecidos pelo Reactome: {nao_achados}")

link = f"https://reactome.org/PathwayBrowser/#/DTAB=AN&ANALYSIS={token}"
display(Markdown(f"**Abra sua análise no navegador do Reactome:** [clique aqui]({link})  \n"
                 "(o link vale por alguns dias; lá você vê as vias pintadas com seus genes)"))

**Pergunta P7** — Abra o link. No "mapa de explosão solar" (*sunburst*, o ícone redondo no canto), quais grandes áreas da biologia ficaram mais coloridas? Clique em *Glycolysis* e observe: o Reactome desenha **reações** (moléculas entrando e saindo de setas), não só "caixinhas de genes". Que vantagem isso traz para quem estuda química de proteínas?

---
# P8. Integração: genes **e** metabólitos no mesmo mapa (≈ 10 min)

Até aqui usamos só **uma camada ômica** (genes). A ideia central da aula é **integrar camadas**: se os genes de uma via mudaram **e** os metabólitos dessa via também mudaram, a evidência fica muito mais forte — são duas medições independentes contando a mesma história.

## P8.1 Em quais vias estão os nossos metabólitos?

In [ ]:
ids_cpd = "+".join("cpd:" + c for c in METABOLITOS["kegg_id"])   # ex.: cpd:C00031+cpd:C00022+...
r = baixar(f"https://rest.kegg.jp/link/pathway/{ids_cpd}")
cpd_via = pd.read_csv(io.StringIO(r.text), sep="\t", header=None, names=["cpd", "via"])
cpd_via["cpd"] = cpd_via["cpd"].str.replace("cpd:", "", regex=False)
# O KEGG devolve vias de referência ("map00010"); trocamos para a versão humana ("hsa00010")
cpd_via["via"] = cpd_via["via"].str.replace("path:map", "hsa", regex=False)
cpd_via = cpd_via[cpd_via["via"].isin(GENES_DA_VIA)]   # só vias que existem em humanos

nome_cpd = dict(zip(METABOLITOS["kegg_id"], METABOLITOS["metabolito"]))
metab_por_via = (cpd_via.groupby("via")["cpd"]
                 .apply(lambda c: ", ".join(sorted(nome_cpd[x] for x in c)))
                 .rename("metabólitos"))
n_metab = cpd_via.groupby("via")["cpd"].nunique().rename("nº metabólitos")

# Junta com o resultado do enriquecimento de genes (P4)
integrado = (ora.set_index("via")
             .join(n_metab).join(metab_por_via)
             .fillna({"nº metabólitos": 0, "metabólitos": ""}))
integrado["nº metabólitos"] = integrado["nº metabólitos"].astype(int)

duas_camadas = integrado[(integrado["FDR"] < 0.05) & (integrado["nº metabólitos"] >= 2)]
print("Vias apoiadas pelas DUAS camadas (genes com FDR < 5% e ≥ 2 metabólitos medidos):")
display(duas_camadas[["nome", "k (na lista)", "FDR", "nº metabólitos", "metabólitos"]])

so_metabolitos = integrado[(integrado["FDR"] >= 0.05) & (integrado["nº metabólitos"] >= 2)]
print("\nVias com ≥ 2 metabólitos alterados, mas SEM enriquecimento de genes:")
display(so_metabolitos[["nome", "k (na lista)", "FDR", "nº metabólitos", "metabólitos"]].head(10))

**Pergunta P8.1** — Quais vias aparecem com apoio das **duas** camadas? Agora olhe a segunda tabela: o **ciclo de Krebs** (*Citrate cycle (TCA cycle)*) tem vários metabólitos alterados (citrato, succinato, 2‑oxoglutarato), mas **não** aparece no enriquecimento de genes. Por quê? Pista: as enzimas **EGLN1/EGLN3** da nossa lista consomem **2‑oxoglutarato e O₂** para marcar o HIF para destruição e liberam **succinato** — mas o KEGG não as desenha no mapa do ciclo de Krebs. Ou seja: **uma camada ômica enxerga o que a outra não enxerga**, e o mapa de vias é um modelo incompleto da célula. Esse é o argumento central a favor da integração.

## P8.2 Pintando o mapa da glicólise

Agora vamos **ver** a via. Primeiro, a imagem "crua" do mapa do KEGG; depois, um link que abre o mesmo mapa **colorido** com os nossos dados.

In [ ]:
VIA = "hsa00010"   # Glicólise / Gliconeogênese. Experimente também "hsa00020" (ciclo de Krebs) ou "hsa04066" (HIF-1)
r = baixar(f"https://rest.kegg.jp/get/{VIA}/image")
display(Image(data=r.content, width=750))

In [ ]:
# Monta as instruções de cor no formato do "KEGG Mapper — Color":
# cada linha = "identificador  cor_de_fundo"
linhas_cor = []
for kegg_id in traducao["kegg_id"].dropna():
    if kegg_id in GENES_DA_VIA[VIA]:
        linhas_cor.append(f"{kegg_id} orange")            # genes da nossa lista em laranja
for _, m in METABOLITOS.iterrows():
    cor = "red" if m["log2FC"] > 0 else "blue"             # metabólito aumentou = vermelho; diminuiu = azul
    linhas_cor.append(f"{m['kegg_id']} {cor}")

print("Copie o bloco abaixo e cole em https://www.kegg.jp/kegg/mapper/color.html")
print("(em 'Search against', escolha hsa; depois clique em Exec):\n")
print("\n".join(linhas_cor))

link_mapa = (f"https://www.kegg.jp/kegg-bin/show_pathway?map={VIA}&multi_query="
             + "%0a".join(l.replace(" ", "+") for l in linhas_cor))
display(Markdown(f"\nOu tente o atalho direto: [mapa {VIA} colorido]({link_mapa})"))

**Pergunta P8.2** — No mapa colorido: os genes laranjas formam uma **sequência contínua** de passos? Onde ficam o lactato (vermelho) e a glicose (azul)? Conte, em 3 frases, a "história metabólica" da célula em hipóxia usando o mapa como roteiro. (Palavra‑chave para pesquisar depois: **efeito Pasteur** e **efeito Warburg**.)

---
# P9. As proteínas como rede: STRING (≈ 8 min)

Vias são desenhos feitos por curadores. Uma visão complementar é a **rede de interações**: cada proteína é um **nó** (ponto) e cada associação é uma **aresta** (linha). Para um engenheiro, é exatamente um **grafo** — como uma rede de comunicações ou de distribuição de energia.

**Sobre o STRING** (<https://string-db.org>; consórcio SIB‑Suíça, Universidade de Copenhague e EMBL): banco de **associações funcionais** entre proteínas de milhares de organismos. Já o usamos na Aula 5. Relembrando: cada ligação tem um **escore de confiança de 0 a 1** que combina sete fontes de evidência (experimentos, bancos curados, mineração de texto, coexpressão, vizinhança genômica, fusão gênica e co‑ocorrência). Usaremos **escore ≥ 0,7 (alta confiança)**.

E o **`networkx`** é uma biblioteca de Python para construir e medir redes: grau de cada nó, caminhos, comunidades etc.

In [ ]:
r = baixar("https://string-db.org/api/tsv/network", metodo="POST", data={
    "identifiers": "\r".join(GENES_HIPOXIA),   # o STRING pede os nomes separados por "\r"
    "species": 9606,                           # código taxonômico de Homo sapiens no NCBI
    "required_score": 700,                     # confiança mínima 0,700
    "caller_identity": "Aula9-PPGED-IME",
})
arestas = pd.read_csv(io.StringIO(r.text), sep="\t")
arestas = arestas[["preferredName_A", "preferredName_B", "score"]].drop_duplicates()
print(f"Ligações de alta confiança entre as proteínas da lista: {len(arestas)}")

# Constrói o grafo
G = nx.Graph()
G.add_nodes_from(GENES_HIPOXIA)                 # inclui todas, mesmo as sem ligação
for _, a in arestas.iterrows():
    G.add_edge(a["preferredName_A"], a["preferredName_B"], peso=a["score"])

grau = dict(G.degree())                          # grau = nº de vizinhos de cada nó
hubs = sorted(grau.items(), key=lambda x: -x[1])[:8]
print("Proteínas mais conectadas (hubs):", ", ".join(f"{p} ({g})" for p, g in hubs))
isoladas = [p for p, g in grau.items() if g == 0]
print(f"Proteínas sem nenhuma ligação ≥ 0,7: {len(isoladas)} → {', '.join(isoladas)}")

In [ ]:
# Desenho da rede. Cor = participa da glicólise no KEGG? Tamanho = grau.
glicolise = {simbolo_de[g] for g in GENES_DA_VIA["hsa00010"] if g in simbolo_de}
cores = ["#e67e22" if p in glicolise else "#5d8aa8" for p in G.nodes()]
tamanhos = [150 + 120 * grau[p] for p in G.nodes()]

plt.figure(figsize=(10, 8))
posicao = nx.spring_layout(G, seed=42, k=0.6)    # "molas": nós ligados se atraem
nx.draw_networkx_edges(G, posicao, alpha=0.3)
nx.draw_networkx_nodes(G, posicao, node_color=cores, node_size=tamanhos)
nx.draw_networkx_labels(G, posicao, font_size=8)
plt.title("Rede STRING (confiança ≥ 0,7) — laranja: genes da glicólise (KEGG)")
plt.axis("off")
plt.show()

link_string = ("https://string-db.org/cgi/network?species=9606&identifiers="
               + "%0d".join(GENES_HIPOXIA))
display(Markdown(f"Versão interativa no site do STRING: [abrir rede]({link_string})"))

**Pergunta P9** — A rede forma um **aglomerado** (*cluster*) bem definido? Ele coincide com os nós laranjas (glicólise)? Quem são os *hubs*? Lembrando o aviso da Aula 5: uma linha no STRING é **associação funcional**, não necessariamente contato físico. Uma proteína isolada aqui é "menos importante" para a hipóxia? (Dica: pense em genes pouco estudados.)

---
# P10. Salvar os resultados (≈ 2 min)

Tabelas em **CSV** abrem no Excel/LibreOffice e podem ir direto para o material suplementar do seu artigo.

In [ ]:
ora.to_csv("aula9_ORA_KEGG_manual.csv", index=False)
res_gp.drop(columns=["parents", "intersections", "evidences"], errors="ignore") \
      .to_csv("aula9_gProfiler.csv", index=False)
reactome.to_csv("aula9_Reactome.csv", index=False)
integrado.reset_index().to_csv("aula9_integracao_genes_metabolitos.csv", index=False)
arestas.to_csv("aula9_rede_STRING.csv", index=False)

arquivos = ["aula9_ORA_KEGG_manual.csv", "aula9_gProfiler.csv", "aula9_Reactome.csv",
            "aula9_integracao_genes_metabolitos.csv", "aula9_rede_STRING.csv"]
print("Arquivos salvos:", *arquivos, sep="\n  ")
try:                                    # no Colab, oferece o download
    from google.colab import files
    for a in arquivos:
        files.download(a)
except ImportError:
    print("(Fora do Colab: os arquivos estão na pasta atual.)")

---
# Conexão com a teoria

| O que você fez neste caderno | Conceito da aula |
|---|---|
| Traduzir símbolos para `hsa:` (P3) | **Mapeamento de identificadores** — a etapa onde genes "somem" |
| `hypergeom.sf(k-1, N, K, n)` (P4) | **Análise de super‑representação (ORA)** e a lógica da urna |
| Escolher `UNIVERSO` | Importância do **background** |
| `benjamini_hochberg` (P4) | **Correção para testes múltiplos (FDR)** |
| Lista aleatória (P5) | **Controle negativo** e falsos positivos |
| g:Profiler com GO, KEGG, Reactome, WP (P6) | Diferentes **bancos de vias** e a **redundância do GO** |
| Diferenças entre nossa análise e o g:Profiler | **Resultados dependem de escolhas e versões** |
| Reactome e seu *token* (P7) | Vias **curadas reação a reação**; hierarquia |
| Metabólitos → vias e mapa colorido (P8) | **Integração multi‑ômica** por mapeamento em vias |
| Grafo do STRING e *hubs* (P9) | **Redes de interação**; associação ≠ contato físico |

## Exercícios (para a lista)

1. **Limiar de confiança.** Na P9, troque `required_score` para `400` e depois `900`. O que acontece com o número de ligações e de proteínas isoladas? Qual escolha você justificaria num artigo?
2. **Outra via no mapa.** Na P8.2, troque `VIA` por `"hsa00020"` (ciclo de Krebs) e por `"hsa04066"` (HIF‑1). Qual mapa conta melhor a história dos metabólitos?
3. **Universo.** Na P4, o universo eram genes com alguma via no KEGG. Num experimento real de proteômica, por que o universo correto seria *"todas as proteínas que o equipamento conseguiu detectar"*? Que erro você cometeria usando o genoma inteiro?
4. **Sua lista.** Substitua `GENES_HIPOXIA` pela lista de proteínas significativas da **Aula 5** (arquivo `resultado_proteomica_cSCC.csv`) ou por uma lista da sua linha de pesquisa. Rode tudo de novo e escreva um parágrafo de "Resultados" no estilo de artigo.
5. **Defesa QBRN.** Monte uma lista com genes da **sinapse colinérgica** relacionados à **acetilcolinesterase (ACHE)** da Aula 7 (ex.: `ACHE, BCHE, CHAT, SLC18A3, SLC5A7, CHRM1, CHRM2, CHRNA7, PON1`), rode a P4 e a P9 e discuta: que vias seriam afetadas por um agente organofosforado?